# Pruebas con los algoritmos · borrador

La cocina de los modelos. Aquí se prueba rápido y en sucio: rejillas de hiperparámetros,
cuánto tarda cada uno, si la red de Keras converge. Lo que salga en limpio se lleva a
`src/model_trainer.py`.

**Este notebook no se defiende y no tiene que estar bonito.** Pero sí se commitea: la
rúbrica valora la visibilidad del trabajo en el repositorio, y esto demuestra que
probaste cosas antes de elegir.

Para no repetirte, usa `data_loader.preparar()` en lugar de volver a cargar el CSV a mano.

In [ ]:
import sys, pathlib

# Subir hasta la raiz del proyecto (la carpeta que contiene src/), la abras desde donde la abras.
raiz = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "src").is_dir())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

from src import data_loader, model_trainer

d = data_loader.preparar()
X_train, y_train = d["X_train"], d["y_train"]
print(X_train.shape)

## 1. Muestra pequeña para iterar rápido

95.512 filas x 5 folds x 6 modelos es una espera larga para probar una idea. Se prueba
con una muestra estratificada y solo al final se lanza el pipeline entero.

In [ ]:
from sklearn.model_selection import train_test_split

X_mini, _, y_mini, _ = train_test_split(
    X_train, y_train, train_size=10_000, stratify=y_train, random_state=config.SEMILLA,
)
print(X_mini.shape, round(y_mini.mean(), 4))

## 2. Cuánto tarda cada modelo

Dato práctico: decide qué entra en el modo `--demo` de la defensa.

In [ ]:
# TODO: cronometrar un fit por modelo sobre X_mini y apuntar los segundos.

## 3. Rejillas de hiperparámetros

Lo que se descubra aquí se copia a `espacio_busqueda()` de cada clase del registro.
Recuerda la sintaxis `paso__hiperparametro`.

In [ ]:
# TODO: RandomizedSearchCV sobre X_mini, un modelo cada vez.
# Apunta el mejor conjunto y, sobre todo, CUANTO mejora respecto a los valores por
# defecto: si la mejora es del 0,3 %, dilo en la reflexion critica en vez de fingir
# que fue la clave del proyecto.

## 4. La red de Keras: la trampa del proyecto

La red se construye en `fit`, **nunca** en `__init__`. Si se construye en `__init__`,
`clone()` reparte el mismo objeto de Keras a los cinco folds y, como el `fit` de Keras no
reinicia los pesos, el fold 2 arranca habiendo visto ya sus datos de validación. El F1
sale inflado y no salta ningún error.

In [ ]:
# TODO: probar aqui la arquitectura (capas, dropout, epocas, early stopping) antes de
# llevarla a la clase RedKeras. Comprueba que dos fits seguidos dan el mismo resultado:
# si el segundo sale mejor, los pesos no se estan reiniciando.

## 5. Balanceo de clases (bonus)

`class_weight="balanced"` es una línea; SMOTE necesita `imbalanced-learn` y va DENTRO del
pipeline, nunca antes de partir.

In [ ]:
# TODO: comparar F1 con y sin class_weight. Si no mejora, dilo igualmente en el README:
# una prueba negativa bien contada puntua.